# Original versus fast Sliced Normal: convergence

First run the environment setup cell, select its new Jupyter kernel, then run
all cells in order. This notebook calls the two **actual public fitters**:
`sliced_normals.legacy_optimization.estimate_optimal_B` and the fast default
`sliced_normals.estimate_optimal_B`.
It compares the objective and Riemannian gradient norm against both iteration
and elapsed time, reports why each fit stopped, and checks final fit quality
using fresh integration samples on the same support.

Each pair uses the same data, degree, uniform integration points, FMLE starting
matrix, penalty, floor, and optimizer limits. The notebook checks the grids and
initial matrices for exact equality after fitting. It does not change production
code, model support, optimizer steps, or the integration rule.

Defaults: bundled 1,000-row data, three integration seeds, degree 2, 5,000 grid
points, up to 10,000 iterations or 120 seconds **per fit**. The time guard bounds
the solver approximately; preprocessing and diagnostics take additional time.
There are six measured fits. A time/iteration/step-size stop is reported separately
from meeting the gradient tolerance. Add the optional synthetic 10,000-row case
in the configuration cell for a larger-data comparison.

No results are prefilled. Output is saved under a new timestamped directory in
`Demo/outputs/convergence_runs/`. Previous timing notebooks are untouched.

## Environment

Run the next cell in your current kernel. It uses only Python's standard library
to create `.venv`, install the pinned dependencies
from `requirements-reproducible.txt`, check imports, and register a Jupyter kernel.
It finds Python 3.11 automatically; set `BASE_PYTHON` to its executable if needed.

When setup finishes, **select `Python (Sliced Normal Fast)` in the notebook's
kernel selector**, then run all cells. This kernel selection activates the new
environment for the notebook. A shell `source .../activate` in a cell cannot
switch the Python process of a running notebook. Setup also prints the shell
activation command for terminal use. Rerunning setup reuses the same environment;
it does not install packages into your original kernel.

References: [Python virtual environments](https://docs.python.org/3.11/library/venv.html)
and [registering an IPython kernel](https://ipython.readthedocs.io/en/stable/install/kernel_install.html).

The tracing hook targets [Pymanopt 2.2.1](https://pymanopt.org/docs/stable/_modules/pymanopt/optimizers/steepest_descent.html).
Its per-iteration log describes the point **before** that iteration's line search.
The returned point is audited separately rather than trusting the result's
gradient norm to describe that final point.

In [ ]:
# Run this cell first. It does not require NumPy, pandas, or the fitting packages.
import os
import sys
import json
import shlex
import shutil
import hashlib
import subprocess
from pathlib import Path

BASE_PYTHON = None  # Optional: '/absolute/path/to/python3.11'
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sliced_normals' / 'optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or one of its subfolders.')
ENV_DIR = ROOT / '.venv'
ENV_PYTHON = ENV_DIR / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
REQUIREMENTS = ROOT / 'requirements-reproducible.txt'
KERNEL_NAME = 'sliced-normal-fast'
KERNEL_LABEL = 'Python (Sliced Normal Fast)'

def run_setup_command(args):
    args = [str(arg) for arg in args]
    print('>', shlex.join(args), flush=True)
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f'Setup command failed with exit code {status}. See the output above.')

def python_version(executable):
    try:
        result = subprocess.run(
            [str(executable), '-c', 'import sys,json; print(json.dumps(list(sys.version_info[:2])))'],
            check=True, capture_output=True, text=True)
        return tuple(json.loads(result.stdout))
    except (OSError, subprocess.SubprocessError, ValueError):
        return None

if not REQUIREMENTS.is_file():
    raise RuntimeError(f'Missing dependency file: {REQUIREMENTS}')
if not ENV_PYTHON.is_file():
    candidates = ([BASE_PYTHON] if BASE_PYTHON else
                  [shutil.which('python3.11'), sys.executable, shutil.which('python3')])
    base = next((candidate for candidate in candidates
                 if candidate and python_version(candidate) == (3, 11)), None)
    if base is None:
        raise RuntimeError('Install Python 3.11, set BASE_PYTHON above to its executable, and rerun this cell.')
    run_setup_command([base, '-m', 'venv', ENV_DIR])
if python_version(ENV_PYTHON) != (3, 11):
    raise RuntimeError(f'{ENV_DIR} is not a working Python 3.11 environment. '
                       'Choose a different ENV_DIR or recreate that environment, then rerun setup.')

run_setup_command([ENV_PYTHON, '-m', 'pip', '--require-virtualenv', 'install', '-r', REQUIREMENTS])
run_setup_command([ENV_PYTHON, '-m', 'pip', 'check'])
run_setup_command([
    ENV_PYTHON, '-c',
    'import sys; sys.path.insert(0, sys.argv[1]); '
    'import numpy,pandas,scipy,autograd,pymanopt,emcee,matplotlib,threadpoolctl,ipykernel; '
    'import sliced_normals,sliced_normals_fast; '
    'print("All notebook and fitter imports passed.")', str(ROOT)])
run_setup_command([ENV_PYTHON, '-m', 'ipykernel', 'install', '--user',
                   '--name', KERNEL_NAME, '--display-name', KERNEL_LABEL])

print('\nEnvironment:', ENV_DIR)
print('Current kernel:', sys.executable)
if Path(sys.prefix).resolve() == ENV_DIR.resolve():
    print('The notebook is already using the new environment. Continue with the next cell.')
else:
    print(f'NEXT: select "{KERNEL_LABEL}" in the notebook kernel selector, then run all cells.')
    print('If it is not listed, refresh the kernel list or choose this interpreter:', ENV_PYTHON)
if os.name != 'nt':
    print('Optional terminal activation: source ' + shlex.quote(str(ENV_DIR / 'bin/activate')))
else:
    print('Optional PowerShell activation: & "' + str(ENV_DIR / 'Scripts/Activate.ps1') + '"')

In [ ]:
import os
import sys
import time
import json
import hashlib
import platform
import importlib
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from unittest.mock import patch

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'sliced_normals' / 'optimization.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or one of its subfolders.')
expected_environment = ROOT / '.venv'
if Path(sys.prefix).resolve() != expected_environment.resolve():
    raise RuntimeError(
        'Switch the notebook kernel to "Python (Sliced Normal Fast)", then run all cells. '
        f'The current kernel is still {sys.executable}. '
        'The environment setup cell creates and registers the required kernel.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import autograd.numpy as anp
from autograd import grad
from autograd.scipy.special import logsumexp as autograd_logsumexp
from scipy.special import logsumexp
from IPython.display import display
from threadpoolctl import threadpool_limits, threadpool_info
from pymanopt.optimizers import SteepestDescent
from pymanopt.manifolds import SymmetricPositiveDefinite

sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
original_module = importlib.import_module('sliced_normals.legacy_optimization')
fast_module = importlib.import_module('sliced_normals.optimization')
from sliced_normals.features import Features, get_F_and_Random_Samples
from sliced_normals_fast import CachedObjective

VERSIONS = {name: importlib.metadata.version(name) for name in
            ['numpy', 'pandas', 'scipy', 'autograd', 'pymanopt', 'emcee',
             'matplotlib', 'threadpoolctl']}
if VERSIONS['pymanopt'] != '2.2.1':
    raise RuntimeError('This tracing hook requires pymanopt==2.2.1. Install it and restart the kernel.')
print('Repository:', ROOT)
print('Python:', platform.python_version(), '|', platform.platform())
print('Dependencies:', VERSIONS)

## Configuration

Start with these defaults. Increase `MAX_ITERATIONS` or `MAX_SECONDS_PER_FIT` if
the corresponding cap is reached. If step size becomes too small, increasing
the iteration limit alone will not resolve the numerical issue. The gradient
tolerance below matches the earlier demo comparison; reduce it for a stricter
comparison. The same settings are applied to both implementations.

`RUN_SEEDS` changes integration points, not the dataset. Comparisons between
seeds therefore measure integration-grid sensitivity on a fixed dataset.
Independent normalization batches assess Monte Carlo sensitivity; they are
**not held-out data** or a proof of normalization accuracy.

In [ ]:
CASE_NAMES = ['bundled_3d']  # Optional: ['bundled_3d', 'synthetic_10k']
RUN_SEEDS = [561, 562, 563]
DEGREE = 2
N_GRID = 5000
MAX_ITERATIONS = 10000
MAX_SECONDS_PER_FIT = 120.0
GRADIENT_TOLERANCE = 5e-4
MIN_STEP_SIZE = 1e-10
PENALTY_LAMBDA = 0.0
WINSOR_THRESHOLD = 1e-10
BLAS_THREADS = 1

# Independent normalization on the fixed training box; shared by each pair.
AUDIT_GRID_COUNT = 50000
AUDIT_BATCHES = 3
AUDIT_CHUNK_SIZE = 10000
AUDIT_SEED = 71842

# Time to reach common objective accuracy, measured relative to the best
# reference-evaluated objective observed at either endpoint (not a known optimum).
OBJECTIVE_TOLERANCES = [1e-2, 1e-3, 1e-4]

assert CASE_NAMES and set(CASE_NAMES) <= {'bundled_3d', 'synthetic_10k'}
assert RUN_SEEDS and len(set(RUN_SEEDS)) == len(RUN_SEEDS)
assert N_GRID > 0 and MAX_ITERATIONS > 0 and MAX_SECONDS_PER_FIT > 0
assert GRADIENT_TOLERANCE > 0 and WINSOR_THRESHOLD > 0
assert AUDIT_BATCHES >= 2 and AUDIT_GRID_COUNT > 0 and AUDIT_CHUNK_SIZE > 0

def synthetic_data(n=10000, seed=124):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=n)
    return pd.DataFrame({
        'X_1': x,
        'X_2': 0.7 * (x*x - 1) + 0.4 * rng.normal(size=n),
        'X_3': rng.normal(size=n),
    })

CASES = {}
for name in CASE_NAMES:
    CASES[name] = (pd.read_csv(ROOT / 'sliced_normals/Sample_Data.csv')
                   [['X_1', 'X_2', 'X_3']] if name == 'bundled_3d'
                   else synthetic_data())
for name, data in CASES.items():
    assert np.isfinite(data.to_numpy()).all()
    print(name, '| rows:', len(data), '| variables:', data.shape[1])

FIT_SETTINGS = dict(degree=DEGREE, n_grid=N_GRID, verbosity=0,
                    max_iterations=MAX_ITERATIONS,
                    min_gradient_norm=GRADIENT_TOLERANCE,
                    penalty_lambda=PENALTY_LAMBDA,
                    winsor_threshold=WINSOR_THRESHOLD)
CONFIG = dict(case_names=CASE_NAMES, seeds=RUN_SEEDS, fit_settings=FIT_SETTINGS,
              max_seconds_per_fit=MAX_SECONDS_PER_FIT, min_step_size=MIN_STEP_SIZE,
              blas_threads=BLAS_THREADS, audit_grid_count=AUDIT_GRID_COUNT,
              audit_batches=AUDIT_BATCHES, audit_chunk_size=AUDIT_CHUNK_SIZE,
              audit_seed=AUDIT_SEED, objective_tolerances=OBJECTIVE_TOLERANCES)
display(pd.Series(CONFIG, name='configuration').to_frame())

## Reference objective and tracing

The reference below reproduces the original default objective, including its
partition floor and penalty. A small preflight checks the cached objective and
analytic derivative against autograd in both floor branches before long runs.

The optimizer subclass only records scalars at its existing logging hook and
sets the same time/step-size guards for both fitters. No iterates are retained.
Elapsed fit time includes feature generation, FMLE, and tracing overhead;
solver time is reported separately. These are instrumented timing comparisons.
The library's `cost_evaluations` field is not used as a count of line-search
evaluations. Compare objectives/densities, since B itself is nonidentifiable.

In [ ]:
def reference_objective(Z, grid, volume, penalty, threshold):
    def cost(B):
        empirical = anp.mean(anp.einsum('ij,jk,ik->i', Z, B, Z))
        energies = anp.einsum('ij,jk,ik->i', grid, B, grid)
        raw = autograd_logsumexp(-energies) - anp.log(len(grid)) + anp.log(volume)
        return (empirical + max(raw, anp.log(threshold))
                + penalty * anp.sqrt(anp.sum(B**2)))
    return cost

rng = np.random.default_rng(81)
Zcheck = Features(pd.DataFrame(rng.normal(size=(80, 2))), 2).to_numpy()
Gcheck = Features(pd.DataFrame(rng.uniform(-2, 2, (150, 2))), 2).to_numpy()
A = rng.normal(size=(Zcheck.shape[1], Zcheck.shape[1]))
Bcheck = 0.02*A.T@A + 0.1*np.eye(len(A))
for threshold in [1e-10, 1e6]:
    ref = reference_objective(Zcheck, Gcheck, 16., 0.07, threshold)
    cached = CachedObjective(Zcheck, Gcheck, 16., 0.07, threshold)
    np.testing.assert_allclose(cached.cost(Bcheck), ref(Bcheck), atol=1e-11, rtol=1e-11)
    np.testing.assert_allclose(cached.gradient(Bcheck), grad(ref)(Bcheck),
                               atol=1e-11, rtol=1e-11)
print('Preflight passed: cost and gradient agree in both floor branches.')

def run_api(module, data, seed, settings, record_trace=True):
    captured = {'history': []}
    get_grid = module.get_F_and_Random_Samples

    def capture_grid(*args, **kwargs):
        result = get_grid(*args, **kwargs)
        captured['Zdf'], captured['Gdf'], captured['volume'] = result
        return result

    class RecordingSteepestDescent(SteepestDescent):
        def __init__(self, *args, **kwargs):
            kwargs.update(max_time=MAX_SECONDS_PER_FIT, min_step_size=MIN_STEP_SIZE)
            super().__init__(*args, **kwargs)

        def run(self, problem, *, initial_point=None, **kwargs):
            # Capture a reference; copy/hash only after the timed API has returned.
            captured['initial'] = initial_point
            captured['solver_start'] = time.perf_counter()
            result = super().run(problem, initial_point=initial_point, **kwargs)
            captured['solver_s'] = time.perf_counter() - captured['solver_start']
            captured['result'] = result
            return result

        def _add_log_entry(self, *, iteration, point, cost, **kwargs):
            if record_trace:
                now = time.perf_counter()
                captured['history'].append(dict(
                    iteration=int(iteration), objective=float(cost),
                    gradient_norm=float(kwargs['gradient_norm']),
                    fit_elapsed_s=now-captured['fit_start'],
                    solver_elapsed_s=now-captured['solver_start']))

    # Restore the notebook's RNG even if a fit raises. Each API gets the same seed.
    state = np.random.get_state()
    try:
        np.random.seed(seed)
        with patch.object(module, 'get_F_and_Random_Samples', capture_grid), \
             patch.object(module, 'SteepestDescent', RecordingSteepestDescent):
            captured['fit_start'] = time.perf_counter()
            B = module.estimate_optimal_B(data, **settings)
            captured['total_s'] = time.perf_counter() - captured['fit_start']
    finally:
        np.random.set_state(state)
    captured['B'] = np.array(B, copy=True)
    captured['initial'] = np.array(captured['initial'], copy=True)
    return captured

def audit_endpoint(run):
    Z = run['Zdf'].to_numpy(dtype=float)
    grid = run['Gdf'].to_numpy(dtype=float)
    B = run['B']
    ref = reference_objective(Z, grid, run['volume'], PENALTY_LAMBDA, WINSOR_THRESHOLD)
    euclidean_grad = grad(ref)(B)
    manifold = SymmetricPositiveDefinite(len(B))
    riemannian_grad = manifold.euclidean_to_riemannian_gradient(B, euclidean_grad)
    raw = logsumexp(-np.einsum('ij,jk,ik->i', grid, B, grid))
    raw += np.log(run['volume']) - np.log(len(grid))
    eigenvalues = np.linalg.eigvalsh((B+B.T)/2)
    if not np.isfinite(B).all() or eigenvalues[0] <= 0:
        raise RuntimeError('Returned matrix is not finite SPD; inspect this run before comparing.')
    return dict(
        final_reference_objective=float(ref(B)),
        final_reference_gradient_norm=float(manifold.norm(B, riemannian_grad)),
        final_euclidean_gradient_norm=float(np.linalg.norm(euclidean_grad)),
        min_eigenvalue=float(eigenvalues[0]),
        condition_number=float(eigenvalues[-1]/eigenvalues[0]),
        raw_log_partition=float(raw),
        floor_active=bool(raw < np.log(WINSOR_THRESHOLD)),
        initial_reference_objective=float(ref(run['initial'])),
    )

## Run paired fits

The warmup is excluded from measured runs. Old/new order alternates between
seeds/cases. Each completed run is saved immediately, so completed fits survive
an interruption. A failed fit raises visibly; it is not silently excluded.

Three integration seeds are not repeated timing trials of the same objective.
Interpret speed ratios together with stationarity, objective agreement, and
stopping reason; ratios for runs stopped at different quality levels do not
measure time to equivalent convergence.

In [ ]:
OUT = ROOT / 'Demo/outputs/convergence_runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT.mkdir(parents=True, exist_ok=False)
source_files = ['sliced_normals/legacy_optimization.py', 'sliced_normals/optimization.py', 'sliced_normals/features.py',
                'sliced_normals/fmle.py', 'sliced_normals_fast/optimization.py']
metadata = dict(config=CONFIG, versions=VERSIONS, python=platform.python_version(),
                platform=platform.platform(), utc_started=datetime.now(timezone.utc).isoformat(),
                source_sha256={p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
                               for p in source_files},
                data_sha256={name: hashlib.sha256(
                    data.to_numpy(dtype=np.float64).tobytes()).hexdigest()
                    for name, data in CASES.items()})
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2)+'\n')
print('Saving results to:', OUT, flush=True)

summary_rows, trace_frames, fitted = [], [], {}
with threadpool_limits(limits=BLAS_THREADS):
    (OUT / 'threadpools.json').write_text(json.dumps(threadpool_info(), indent=2)+'\n')
    warm_settings = dict(FIT_SETTINGS, max_iterations=3, n_grid=300)
    warm_data = next(iter(CASES.values())).iloc[:120]
    for module in [original_module, fast_module]:
        run_api(module, warm_data, 59, warm_settings, record_trace=False)
    print('Warmup complete.', flush=True)

    for case_index, (case, data) in enumerate(CASES.items()):
        for seed_index, seed in enumerate(RUN_SEEDS):
            order = [('original', original_module), ('fast', fast_module)]
            if (case_index + seed_index) % 2:
                order.reverse()
            signatures = {}
            for variant, module in order:
                print(f'Running {case}, seed={seed}, {variant} ...', flush=True)
                run = run_api(module, data, seed, FIT_SETTINGS)
                # All auditing is outside the timed fitter call.
                audit = audit_endpoint(run)
                result = run['result']
                signatures[variant] = (
                    run['Zdf'].to_numpy(copy=True), run['Gdf'].to_numpy(copy=True),
                    run['volume'], run['initial'].copy())
                frame = pd.DataFrame(run['history'])
                frame.insert(0, 'variant', variant)
                frame.insert(0, 'seed', seed)
                frame.insert(0, 'case', case)
                # Append the actual returned endpoint with a freshly computed gradient.
                frame['is_final_point'] = False
                endpoint = dict(case=case, seed=seed, variant=variant,
                                iteration=int(result.iterations)+1,
                                objective=audit['final_reference_objective'],
                                gradient_norm=audit['final_reference_gradient_norm'],
                                fit_elapsed_s=run['total_s'], solver_elapsed_s=run['solver_s'],
                                is_final_point=True)
                frame = pd.concat([frame, pd.DataFrame([endpoint])], ignore_index=True)
                trace_frames.append(frame)
                np.save(OUT / f'{case}_seed{seed}_{variant}_B.npy', run['B'])
                np.save(OUT / f'{case}_seed{seed}_initial_B.npy', run['initial'])
                row = dict(case=case, seed=seed, variant=variant, n=len(data),
                           q=len(run['B']), iterations=int(result.iterations),
                           total_s=run['total_s'], solver_s=run['solver_s'],
                           preprocessing_s=run['solver_start']-run['fit_start'],
                           stopping_reason=result.stopping_criterion,
                           library_reported_gradient_norm=float(result.gradient_norm),
                           gradient_tolerance_met=bool(
                               audit['final_reference_gradient_norm'] < GRADIENT_TOLERANCE),
                           **audit)
                summary_rows.append(row)
                fitted[(case, seed, variant)] = run['B']
                pd.DataFrame(summary_rows).to_csv(OUT/'fit_summary.csv', index=False)
                pd.concat(trace_frames, ignore_index=True).to_csv(OUT/'convergence_traces.csv', index=False)
                print(f"  {result.stopping_criterion} Final checked gradient="
                      f"{row['final_reference_gradient_norm']:.3g}; total={row['total_s']:.3f}s",
                      flush=True)
                del run
            old, new = signatures['original'], signatures['fast']
            for index in [0, 1, 3]:
                np.testing.assert_array_equal(old[index], new[index])
            assert old[2] == new[2]
            print('  Verified identical features, integration grid, volume, and FMLE start.', flush=True)
            del signatures

summary = pd.DataFrame(summary_rows)
traces = pd.concat(trace_frames, ignore_index=True)
display(summary[['case', 'seed', 'variant', 'iterations', 'total_s',
                 'final_reference_objective', 'final_reference_gradient_norm',
                 'gradient_tolerance_met', 'floor_active', 'condition_number', 'stopping_reason']])

## Convergence curves

Each row of panels compares one fixed integration grid. The left column measures
progress per iteration; the right measures progress per wall-clock second,
including preprocessing. The appended endpoint uses `iterations + 1` on the
iteration axis to distinguish it from the pre-line-search log of the last
iteration. Dashed horizontal lines show the requested gradient tolerance.

Riemannian gradient norms can become small near a poorly conditioned SPD boundary.
Inspect condition numbers, Euclidean gradients, floor activation, and independent
fit-quality diagnostics as well. A small gradient is not a global-optimality or
statistical-accuracy certificate.

In [ ]:
for (case, seed), pair in traces.groupby(['case', 'seed'], sort=False):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained')
    for variant, group in pair.groupby('variant', sort=False):
        for col, x in enumerate(['iteration', 'fit_elapsed_s']):
            axes[0, col].plot(group[x], group.objective, label=variant)
            axes[1, col].semilogy(group[x], np.maximum(group.gradient_norm, 1e-300), label=variant)
            axes[0, col].set_xlabel('Iteration / appended endpoint' if col == 0 else 'Elapsed fit time (s)')
            axes[1, col].set_xlabel(axes[0, col].get_xlabel())
    for col in range(2):
        axes[0, col].set_ylabel('Fitting objective (lower is better)')
        axes[1, col].set_ylabel('Riemannian gradient norm')
        axes[1, col].axhline(GRADIENT_TOLERANCE, color='gray', linestyle='--', label='tolerance')
    for ax in axes.flat:
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle(f'{case}, integration seed {seed}')
    fig.savefig(OUT/f'{case}_seed{seed}_convergence.png', dpi=150)
    plt.show()

## Time to a common objective level

For each pair, the baseline is the better of the two reference-evaluated endpoint
objectives. It is an **observed reference, not the true optimum**. Targets are
that value plus the absolute tolerances configured above. The table reports the
first recorded iterate at or below each target. Missing times mean the target
was not observed within the budget; they are not replaced by the run duration.
Also compare times to the gradient tolerance and final stationarity directly.

In [ ]:
target_rows = []
for (case, seed), pair_summary in summary.groupby(['case', 'seed'], sort=False):
    best_endpoint = pair_summary.final_reference_objective.min()
    pair_trace = traces[(traces.case == case) & (traces.seed == seed)]
    for tolerance in OBJECTIVE_TOLERANCES:
        target = best_endpoint + tolerance
        for variant in ['original', 'fast']:
            group = pair_trace[pair_trace.variant == variant]
            hits = group[group.objective <= target]
            target_rows.append(dict(case=case, seed=seed, variant=variant,
                                    absolute_objective_tolerance=tolerance,
                                    best_observed_endpoint=best_endpoint, target=target,
                                    reached=not hits.empty,
                                    first_fit_seconds=float(hits.fit_elapsed_s.iloc[0]) if len(hits) else np.nan,
                                    first_logged_iteration=int(hits.iteration.iloc[0]) if len(hits) else np.nan))
targets = pd.DataFrame(target_rows)
targets.to_csv(OUT/'time_to_objective.csv', index=False)
display(targets)

gradient_hits = []
for (case, seed, variant), group in traces.groupby(['case', 'seed', 'variant'], sort=False):
    hits = group[group.gradient_norm < GRADIENT_TOLERANCE]
    gradient_hits.append(dict(case=case, seed=seed, variant=variant,
                              first_tolerance_seconds=float(hits.fit_elapsed_s.iloc[0]) if len(hits) else np.nan,
                              tolerance_observed=not hits.empty))
gradient_times = pd.DataFrame(gradient_hits)
gradient_times.to_csv(OUT/'time_to_gradient_tolerance.csv', index=False)
display(gradient_times)

paired = summary.pivot(index=['case', 'seed'], columns='variant',
                       values=['total_s', 'final_reference_objective',
                               'final_reference_gradient_norm', 'gradient_tolerance_met'])
comparison_rows = []
for (case, seed), row in paired.iterrows():
    comparison_rows.append(dict(
        case=case, seed=seed,
        elapsed_ratio_original_over_fast=row[('total_s', 'original')]/row[('total_s', 'fast')],
        objective_fast_minus_original=row[('final_reference_objective', 'fast')]-row[('final_reference_objective', 'original')],
        both_final_gradients_below_tolerance=bool(
            row[('gradient_tolerance_met', 'original')] and row[('gradient_tolerance_met', 'fast')]),
    ))
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUT/'paired_comparison.csv', index=False)
display(comparison)

## Independent normalization and density agreement

Use fresh uniform grids in the **training box**, shared between the final fits.
No normalization floor or fitting penalty is used in these likelihood diagnostics.
Report approximate mean log likelihood on the training observations and variation
across independent normalization batches. This is in-sample assessment with fresh
integration samples, not held-out validation.

Grid Hellinger distance compares normalized quadrature weights of the two fits
on each fresh grid (zero means agreement on that grid). It approximates a density
distance but cannot detect mass missed by both fits' quadrature. Grid effective
sample size and largest normalized weight help expose concentration; they do not
prove integration accuracy. Low ESS or unstable log likelihoods call for a larger
grid before drawing conclusions. This does not test the unchanged MCMC sampler.

In [ ]:
def grid_energies(points, B):
    chunks = []
    for start in range(0, len(points), AUDIT_CHUNK_SIZE):
        Z = Features(pd.DataFrame(points[start:start+AUDIT_CHUNK_SIZE]), DEGREE).to_numpy()
        chunks.append(np.einsum('ij,jk,ik->i', Z, B, Z))
    return np.concatenate(chunks)

quality_rows, density_rows = [], []
with threadpool_limits(limits=BLAS_THREADS):
    for case_index, (case, data) in enumerate(CASES.items()):
        low = data.min().to_numpy() - 0.01*np.abs(data.min().to_numpy())
        high = data.max().to_numpy() + 0.01*np.abs(data.max().to_numpy())
        volume = np.prod(high-low)
        train_Z = Features(data, DEGREE).to_numpy()
        empirical_energies = {
            (seed, variant): float(np.mean(np.einsum(
                'ij,jk,ik->i', train_Z, fitted[(case, seed, variant)], train_Z)))
            for seed in RUN_SEEDS for variant in ['original', 'fast']}
        del train_Z
        for batch in range(AUDIT_BATCHES):
            rng = np.random.default_rng(np.random.SeedSequence([AUDIT_SEED, case_index, batch]))
            points = rng.uniform(low, high, size=(AUDIT_GRID_COUNT, len(low)))
            for seed in RUN_SEEDS:
                log_weights = {}
                for variant in ['original', 'fast']:
                    energy = grid_energies(points, fitted[(case, seed, variant)])
                    log_sum = logsumexp(-energy)
                    log_partition = log_sum - np.log(len(points)) + np.log(volume)
                    lw = -energy-log_sum
                    log_weights[variant] = lw
                    quality_rows.append(dict(
                        case=case, seed=seed, batch=batch, variant=variant,
                        mean_log_likelihood=-empirical_energies[(seed, variant)]-log_partition,
                        log_partition=float(log_partition),
                        grid_ess=float(np.exp(-logsumexp(2*lw))),
                        max_grid_weight=float(np.exp(lw.max())),
                    ))
                affinity = np.exp(logsumexp(0.5*(log_weights['original']+log_weights['fast'])))
                density_rows.append(dict(case=case, seed=seed, batch=batch,
                                         grid_hellinger=float(np.sqrt(np.clip(1-affinity, 0, 1)))))
            print(f'{case}: independent normalization batch {batch+1}/{AUDIT_BATCHES} complete.', flush=True)
            pd.DataFrame(quality_rows).to_csv(OUT/'independent_normalization.csv', index=False)
            pd.DataFrame(density_rows).to_csv(OUT/'density_agreement.csv', index=False)

quality = pd.DataFrame(quality_rows)
density = pd.DataFrame(density_rows)
quality_summary = quality.groupby(['case', 'seed', 'variant']).agg(
    mean_log_likelihood=('mean_log_likelihood', 'mean'),
    normalization_batch_sd=('mean_log_likelihood', 'std'),
    minimum_grid_ess=('grid_ess', 'min'),
    maximum_grid_weight=('max_grid_weight', 'max'))
quality_summary.to_csv(OUT/'independent_quality_summary.csv')
display(quality_summary)
display(density.groupby(['case', 'seed']).grid_hellinger.agg(['mean', 'max']))

# Paired differences use exactly the same audit points, reducing integration noise.
ll_pairs = quality.pivot(index=['case', 'seed', 'batch'], columns='variant', values='mean_log_likelihood')
ll_pairs['fast_minus_original'] = ll_pairs['fast']-ll_pairs['original']
ll_pairs.to_csv(OUT/'paired_likelihood_differences.csv')
display(ll_pairs.groupby(['case', 'seed']).fast_minus_original.agg(['mean', 'std']))

## Reading the result

- Check stop reasons and **freshly recomputed endpoint gradients** first. Reaching
  a cap or a tiny step does not establish convergence to the requested tolerance.
- Similar objective-versus-iteration curves with faster progress against time
  support a computational speedup. Divergent curves can result from roundoff
  changing line-search choices; compare endpoints and density diagnostics.
- Use the common-objective timing table to compare equivalent observed progress.
  A smaller objective on one fixed grid does not guarantee a better true density.
- Inspect integration-seed sensitivity, independent likelihood variation, grid
  ESS, floor activation, and matrix conditioning. More iterations cannot repair
  an inaccurate normalization grid. Near-boundary stationarity is not a convex
  optimality certificate for this manifold optimization.
- Only describe a timing ratio as time to comparable convergence when endpoint
  quality and stationarity support that interpretation. Otherwise report it as
  elapsed time under the stated budgets.

The output directory contains metadata, source/data hashes, all scalar traces,
summary tables, final/initial matrices, and plots. Generated outputs stay under the ignored `Demo/outputs/` directory.

In [ ]:
print('Results directory:', OUT)
print('Final gradient tolerance met by', int(summary.gradient_tolerance_met.sum()),
      'of', len(summary), 'fits.')
display(summary.groupby(['case', 'variant']).agg(
    runs=('seed', 'count'),
    endpoint_tolerance_hits=('gradient_tolerance_met', 'sum'),
    median_total_seconds=('total_s', 'median'),
    median_final_objective=('final_reference_objective', 'median'),
    worst_final_gradient=('final_reference_gradient_norm', 'max')))